# 02 · Modelos, validación y MLflow

Se comparan **regresión lineal, KNN, Random Forest, XGBoost y SVM**, además de **MLP base y tres experimentos**. Son implementaciones oficiales de sklearn/XGBoost, no algoritmos manuales.

Métrica principal: MAE en euros de renta municipal. Un error de 50 EUR afecta a la referencia que recibe el servicio de orientación, pero no implica que una persona pueda o no recibir una ayuda. También registramos RMSE, R², tiempo y métricas de entrenamiento.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
sys.path.insert(0, str(ROOT))
OUT = Path(os.getenv('RETO1_ARTIFACT_DIR', str(ROOT / 'entrega_ml')))
from src.ml_workflow import train_official
from src.mlflow_tracker import MLflowTracker
if not (OUT / 'models/selection.json').exists():
    train_official(ROOT / 'data/capturas_oficiales', OUT)
else:
    print('Selección ya cerrada: se consultan experimentos existentes, sin reentrenar tras mirar test.')

## Preprocesado y validación

Todos los regresores reciben el mismo `ColumnTransformer`: imputación por mediana y `StandardScaler` para numéricas; imputación y `OneHotEncoder` para categóricas. El pipeline completo queda guardado. La variable objetivo se estandariza con `TransformedTargetRegressor` y las predicciones vuelven a euros.

`GridSearchCV` usa ventanas temporales crecientes sobre train; un trimestre completo pertenece al mismo lado de cada corte. Se comparan vecinos en KNN, hojas mínimas en RF, profundidad en XGBoost y C en SVM. La validación 2024 elige el ganador. En empates dentro del 1% se prefiere regresión lineal. El test no interviene.

In [ ]:
tracker = MLflowTracker(tracking_dir=OUT / 'mlruns')
runs = tracker.results()
display(runs[['run_id', 'tags.mlflow.runName', 'metrics.train_mae', 'metrics.val_mae', 'metrics.val_rmse', 'metrics.val_r2', 'metrics.tiempo_s']])
comparativa = pd.read_csv(OUT / 'resultados/comparativa_mlflow.csv')
display(comparativa)
seleccion = json.loads((OUT / 'models/selection.json').read_text())
display(seleccion)

## Experimentos MLP y justificación

Base: una capa de 32 neuronas ReLU. Experimento 1: (64, 32), para comprobar si mayor capacidad mejora la validación. Experimento 2: 32 neuronas y tanh, cambiando activación. Experimento 3: 32 neuronas y alpha=0.01, aumentando regularización. Se mantiene semilla 42 y un máximo de 1.000 iteraciones.

Los avisos de entrenamiento quedan contabilizados; un warning de convergencia no se oculta ni equivale a un resultado validado. Hay que contrastar train/validación para detectar sobreajuste. La renta del trimestre anterior es un baseline obligatorio, calculado donde el retardo existe.

## SMOTE: ámbito correcto

SMOTE no se aplica a una renta continua. Como ampliación se compara una clasificación auxiliar de **subida trimestral observada** con y sin SMOTE, usando solo variables numéricas escaladas para evitar categorías fraccionarias. Se balancea exclusivamente train; validación mantiene su distribución. La matriz de confusión, precisión, recall y F1 proceden de validación. Este experimento no puntúa la viabilidad de personas ni determina la ayuda.

In [ ]:
display(json.loads((OUT / 'resultados/comparacion_smote.json').read_text()))

## Evaluación final

El modelo elegido se conserva en `models/best_model.joblib`, incluyendo preprocesamiento. La pestaña **Test reservado** de Streamlit ejecuta una única evaluación y conserva sus huellas y resultados. Este notebook no abre ni puntúa ese conjunto. Las capturas deben corresponder al MLflow y a la app realmente ejecutados; no se inventan pantallas ni cifras.